# Visualize Building Thresholds in t-SNE Latent Space

This notebook overlays classified building thresholds (b1-b4) on top of the **existing t-SNE visualization** from train_mae_full.ipynb.

**Approach:**
- Load the exact t-SNE coordinates from train_mae_full.ipynb (ground truth typologies)
- Re-extract embeddings for the same validation samples
- Extract embeddings for building thresholds
- Position buildings in 2D using K-nearest neighbor interpolation (preserves original typology positions)
- Export combined visualization as SVG

## Setup

In [ ]:
import sys
sys.path.append('../models_MAE')

import os
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from sklearn.neighbors import NearestNeighbors

from config_mae import get_mae_config
from timesformer_mae import TimeSformerMAE
from dataset_mae import create_mae_dataloaders
from dataset_classifier import BuildingDataset

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    device = torch.device('cuda')
else:
    device = torch.device('cpu')
print(f"Device: {device}")

## Configuration

In [ ]:
# Paths - Update these to match your local setup
DATA_ROOT = '../data'

# Existing t-SNE coordinates from train_mae_full.ipynb
TSNE_COORDS_PATH = '../models_MAE/output_mae_adjusted/tsne_coordinates.csv'

# Model weights
CHECKPOINT_PATH = '../models_MAE/output_mae_adjusted/checkpoints/best_model.pt'
ENCODER_PATH = '../models_MAE/output_mae_adjusted/encoder_mae_pretrained.pt'

OUTPUT_DIR = '../output_classifier/tsne_buildings'

# Building data
PANOS_BUILDINGS_DIR = os.path.join(DATA_ROOT, 'panos_buildings')
CANDIDATES_DIR = os.path.join(DATA_ROOT, 'candidates')
BUILDING_IDS = ['b1', 'b2', 'b3', 'b4']

# Typology names
TYPOLOGY_NAMES = ['t1', 't2', 't3', 't4', 't5', 't6', 't7', 't8']

# KNN parameters for interpolating building positions
K_NEIGHBORS = 10

# Same train/val split as train_mae_full.ipynb
TRAIN_SPLIT = 0.85
BATCH_SIZE = 32
NUM_WORKERS = 4

os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"Configuration:")
print(f"  Data root: {DATA_ROOT}")
print(f"  Existing t-SNE: {TSNE_COORDS_PATH}")
print(f"  Checkpoint: {CHECKPOINT_PATH}")
print(f"  Buildings: {BUILDING_IDS}")
print(f"  K neighbors: {K_NEIGHBORS}")
print(f"  Output: {OUTPUT_DIR}")

## Load Existing t-SNE Coordinates

In [ ]:
print("Loading existing t-SNE coordinates from train_mae_full.ipynb...")

# Load the saved t-SNE coordinates
tsne_df = pd.read_csv(TSNE_COORDS_PATH)

print(f"✓ Loaded {len(tsne_df)} typology points")
print(f"  Columns: {list(tsne_df.columns)}")
print()

# Extract coordinates and labels
typology_2d = tsne_df[['x', 'y']].values  # (N, 2)
typology_labels = tsne_df['true_label'].values  # (N,)
typology_names_list = tsne_df['typology'].values.tolist()  # ['t1', 't2', ...]

print(f"Typology distribution:")
for typ in TYPOLOGY_NAMES:
    count = sum(1 for t in typology_names_list if t == typ)
    print(f"  {typ}: {count} points")
print()

print(f"✓ These are the EXACT positions from train_mae_full.ipynb")
print()

## Load MAE Encoder

In [ ]:
print("Loading MAE encoder...")

# Load config and model
config = get_mae_config()
model = TimeSformerMAE(config)

# Try to load full checkpoint first, fall back to encoder-only weights
if os.path.exists(CHECKPOINT_PATH):
    print(f"Loading full checkpoint from: {CHECKPOINT_PATH}")
    checkpoint = torch.load(CHECKPOINT_PATH, map_location=device, weights_only=False)
    model.load_state_dict(checkpoint['model_state_dict'])
    print(f"  Trained epoch: {checkpoint['epoch']+1}")
    print(f"  Val loss: {checkpoint['val_loss']:.4f}")
elif os.path.exists(ENCODER_PATH):
    print(f"Loading encoder weights from: {ENCODER_PATH}")
    encoder_weights = torch.load(ENCODER_PATH, map_location=device, weights_only=False)
    model.load_state_dict(encoder_weights, strict=False)
    print(f"  Loaded encoder-only weights")
else:
    raise FileNotFoundError(f"Neither {CHECKPOINT_PATH} nor {ENCODER_PATH} found!")

model = model.to(device)
model.eval()

print(f"✓ Model loaded successfully!")
print()

## Re-extract Validation Embeddings

We need to re-extract embeddings for the same validation samples that were used to create the t-SNE coordinates, so we can use them as reference points for KNN interpolation.

In [ ]:
print("="*80)
print("RE-EXTRACTING VALIDATION EMBEDDINGS")
print("="*80)
print()

# Create the same dataset split as train_mae_full.ipynb
train_loader, val_loader, full_dataset, train_indices, val_indices = create_mae_dataloaders(
    data_root=DATA_ROOT,
    batch_size=BATCH_SIZE,
    train_split=TRAIN_SPLIT,
    num_workers=NUM_WORKERS,
)

print(f"\nValidation set size: {len(val_indices)}")
print(f"Expected from t-SNE CSV: {len(tsne_df)}")

if len(val_indices) != len(tsne_df):
    print(f"⚠ WARNING: Size mismatch! Proceeding anyway...")
print()

# Extract embeddings for validation set (same order as val_indices)
typology_embeddings = []

print("Extracting validation embeddings...")
with torch.no_grad():
    for idx in tqdm(val_indices):
        sample = full_dataset[idx]
        frames = sample['frames'].unsqueeze(0).to(device)  # (1, 7, 1, 32, 64)
        
        # Run encoder WITHOUT masking
        x_encoded, _, _ = model.forward_encoder(frames, frame_mask_ratios=(0,0,0,0,0,0,0))
        
        # Global average pooling
        h = x_encoded.mean(dim=1)  # (1, 384)
        
        typology_embeddings.append(h.cpu().numpy())

typology_embeddings = np.concatenate(typology_embeddings, axis=0)  # (N, 384)

print(f"\n✓ Extracted {len(typology_embeddings)} validation embeddings")
print(f"  Shape: {typology_embeddings.shape}")
print()

## Load Building Dataset

In [ ]:
print("Loading building thresholds...")

building_dataset = BuildingDataset(
    panos_dir=PANOS_BUILDINGS_DIR,
    candidates_dir=CANDIDATES_DIR,
    building_ids=BUILDING_IDS
)

print(f"\n✓ Loaded {len(building_dataset)} building threshold sequences")
print()

## Extract Building Embeddings

In [ ]:
print("="*80)
print("EXTRACTING BUILDING EMBEDDINGS")
print("="*80)
print()

# Load predictions to get classified typologies
predictions_path = '../output_classifier/building_predictions/all_predictions.csv'
predictions_df = pd.read_csv(predictions_path)

print(f"Loaded {len(predictions_df)} building predictions")
print()

# Extract embeddings for all building thresholds
building_embeddings = []
building_ids_list = []
building_threshold_ids = []
building_predicted_classes = []

print("Extracting embeddings...")
with torch.no_grad():
    for idx in tqdm(range(len(building_dataset))):
        sample = building_dataset[idx]
        frames = sample['frames'].unsqueeze(0).to(device)  # (1, 7, 1, 32, 64)
        
        # Run encoder WITHOUT masking
        x_encoded, _, _ = model.forward_encoder(frames, frame_mask_ratios=(0,0,0,0,0,0,0))
        
        # Global average pooling
        h = x_encoded.mean(dim=1)  # (1, 384)
        
        building_embeddings.append(h.cpu().numpy())
        building_ids_list.append(sample['building_id'])
        building_threshold_ids.append(sample['threshold_id'])
        
        # Get predicted class from predictions CSV
        pred_row = predictions_df[predictions_df['threshold_id'] == sample['threshold_id']]
        if len(pred_row) > 0:
            building_predicted_classes.append(pred_row.iloc[0]['predicted_class'])
        else:
            building_predicted_classes.append('unknown')

building_embeddings = np.concatenate(building_embeddings, axis=0)

print(f"\n✓ Extracted {len(building_embeddings)} building embeddings")
print(f"  Shape: {building_embeddings.shape}")
print()

# Summary per building
for building_id in BUILDING_IDS:
    count = sum(1 for bid in building_ids_list if bid == building_id)
    print(f"  {building_id}: {count} thresholds")
print()

## Position Buildings using KNN Interpolation

Since t-SNE is non-parametric, we can't directly project new points. Instead, we use K-nearest neighbor interpolation: for each building embedding, find its K nearest neighbors among the typology embeddings in the high-dimensional space, then compute a weighted average of their 2D positions.

In [ ]:
print("="*80)
print("POSITIONING BUILDINGS IN t-SNE SPACE")
print("="*80)
print()

# Fit KNN on typology embeddings
print(f"Fitting KNN with K={K_NEIGHBORS} neighbors...")
knn = NearestNeighbors(n_neighbors=K_NEIGHBORS, metric='cosine')
knn.fit(typology_embeddings)

# Find nearest neighbors for each building
print("Finding nearest neighbors for buildings...")
distances, indices = knn.kneighbors(building_embeddings)

# Interpolate 2D positions using weighted average
# Weights are inverse of distance (closer neighbors have more influence)
building_2d = []

for i in range(len(building_embeddings)):
    neighbor_indices = indices[i]
    neighbor_distances = distances[i]
    
    # Convert distances to weights (inverse distance weighting)
    # Add small epsilon to avoid division by zero
    weights = 1.0 / (neighbor_distances + 1e-8)
    weights = weights / weights.sum()  # Normalize to sum to 1
    
    # Get 2D coordinates of neighbors
    neighbor_coords = typology_2d[neighbor_indices]  # (K, 2)
    
    # Weighted average
    interpolated_pos = np.sum(neighbor_coords * weights[:, np.newaxis], axis=0)
    building_2d.append(interpolated_pos)

building_2d = np.array(building_2d)  # (M, 2)

print(f"\n✓ Positioned {len(building_2d)} buildings in t-SNE space")
print(f"  Shape: {building_2d.shape}")
print()

# Verify positions are within the typology range
print(f"Position ranges:")
print(f"  Typology X: [{typology_2d[:, 0].min():.2f}, {typology_2d[:, 0].max():.2f}]")
print(f"  Typology Y: [{typology_2d[:, 1].min():.2f}, {typology_2d[:, 1].max():.2f}]")
print(f"  Building X: [{building_2d[:, 0].min():.2f}, {building_2d[:, 0].max():.2f}]")
print(f"  Building Y: [{building_2d[:, 1].min():.2f}, {building_2d[:, 1].max():.2f}]")
print()

## Visualize: Original Typologies + Building Overlay

In [ ]:
print("="*80)
print("CREATING VISUALIZATION")
print("="*80)
print()

# Color palette for typologies (same as train_mae_full.ipynb)
typology_colors = plt.cm.tab10(np.linspace(0, 1, 10))[:8]

# Markers for buildings
building_markers = {
    'b1': 's',  # square
    'b2': '^',  # triangle up
    'b3': 'D',  # diamond
    'b4': 'v'   # triangle down
}

fig, ax = plt.subplots(1, 1, figsize=(16, 12))

# Plot original typologies (ground truth - EXACT same as train_mae_full.ipynb)
for i, typ_name in enumerate(TYPOLOGY_NAMES):
    mask = np.array([t == typ_name for t in typology_names_list])
    if mask.sum() > 0:
        ax.scatter(
            typology_2d[mask, 0],
            typology_2d[mask, 1],
            c=[typology_colors[i]],
            label=f'{typ_name}',
            alpha=0.6,
            s=50,
            edgecolors='black',
            linewidths=0.5,
            marker='o'
        )

# Plot buildings colored by predicted typology, shaped by building ID
for building_id in BUILDING_IDS:
    for pred_class in TYPOLOGY_NAMES:
        mask = np.array([(bid == building_id and pred == pred_class) 
                for bid, pred in zip(building_ids_list, building_predicted_classes)])
        if mask.sum() > 0:
            typ_idx = TYPOLOGY_NAMES.index(pred_class)
            ax.scatter(
                building_2d[mask, 0],
                building_2d[mask, 1],
                c=[typology_colors[typ_idx]],
                alpha=0.9,
                s=120,
                edgecolors='black',
                linewidths=1.5,
                marker=building_markers[building_id]
            )

# Add building markers to legend
for building_id in BUILDING_IDS:
    ax.scatter([], [], c='gray', marker=building_markers[building_id], 
               s=120, label=f'{building_id.upper()}', edgecolors='black', linewidths=1.5)

ax.set_xlabel('t-SNE Dimension 1', fontsize=14, fontweight='bold')
ax.set_ylabel('t-SNE Dimension 2', fontsize=14, fontweight='bold')
ax.set_title(
    'Building Thresholds Overlaid on Original t-SNE\n(Color = Typology, Shape = Building)',
    fontsize=16,
    fontweight='bold',
    pad=20
)
ax.legend(loc='best', fontsize=11, framealpha=0.95, edgecolor='black', ncol=2)
ax.grid(True, alpha=0.3)
ax.set_facecolor('#f8f9fa')

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'tsne_buildings_overlay.png'), dpi=150, bbox_inches='tight')
plt.show()

print(f"\n✓ Saved to: {os.path.join(OUTPUT_DIR, 'tsne_buildings_overlay.png')}")
print()

## Export as SVG

In [ ]:
print("="*80)
print("EXPORTING SVG")
print("="*80)
print()

# Use the same coordinate range as the original t-SNE
x_min, x_max = typology_2d[:, 0].min(), typology_2d[:, 0].max()
y_min, y_max = typology_2d[:, 1].min(), typology_2d[:, 1].max()

# Add some padding
x_range = x_max - x_min
y_range = y_max - y_min
x_min -= 0.05 * x_range
x_max += 0.05 * x_range
y_min -= 0.05 * y_range
y_max += 0.05 * y_range

# SVG dimensions
svg_width = 1000
svg_height = 800
padding = 50

def normalize_coord(x, y):
    nx = padding + (x - x_min) / (x_max - x_min) * (svg_width - 2*padding)
    ny = padding + (y - y_min) / (y_max - y_min) * (svg_height - 2*padding)
    return nx, ny

# Color mapping for typologies (hex)
typology_hex_colors = {
    't1': '#1f77b4',
    't2': '#ff7f0e',
    't3': '#2ca02c',
    't4': '#d62728',
    't5': '#9467bd',
    't6': '#8c564b',
    't7': '#e377c2',
    't8': '#7f7f7f'
}

# Create SVG content
svg_content = f'''<?xml version="1.0" encoding="UTF-8"?>
<svg xmlns="http://www.w3.org/2000/svg" width="{svg_width}" height="{svg_height}" viewBox="0 0 {svg_width} {svg_height}">
  <rect width="100%" height="100%" fill="#f8f9fa"/>
  
  <!-- Original Typologies (circles) - EXACT positions from train_mae_full.ipynb -->
  <g id="typologies">
'''

# Add typology points
for i in range(len(typology_2d)):
    x, y = normalize_coord(typology_2d[i, 0], typology_2d[i, 1])
    typ_name = typology_names_list[i]
    color = typology_hex_colors[typ_name]
    svg_content += f'    <circle cx="{x:.2f}" cy="{y:.2f}" r="4" fill="{color}" stroke="black" stroke-width="0.5" opacity="0.6" data-typology="{typ_name}" data-label="{typology_labels[i]}"/>\n'

svg_content += '  </g>\n\n  <!-- Building Thresholds (positioned via KNN interpolation) -->\n'

# Add building points with different shapes
for building_id in BUILDING_IDS:
    svg_content += f'  <g id="{building_id}">\n'
    
    for i in range(len(building_2d)):
        if building_ids_list[i] == building_id:
            x, y = normalize_coord(building_2d[i, 0], building_2d[i, 1])
            pred_class = building_predicted_classes[i]
            color = typology_hex_colors.get(pred_class, '#000000')
            threshold_id = building_threshold_ids[i]
            
            # Different shapes for different buildings
            if building_id == 'b1':  # square
                svg_content += f'    <rect x="{x-5:.2f}" y="{y-5:.2f}" width="10" height="10" fill="{color}" stroke="black" stroke-width="1.5" opacity="0.9" data-building="{building_id}" data-threshold="{threshold_id}" data-predicted="{pred_class}"/>\n'
            elif building_id == 'b2':  # triangle up
                svg_content += f'    <polygon points="{x:.2f},{y-6:.2f} {x-6:.2f},{y+5:.2f} {x+6:.2f},{y+5:.2f}" fill="{color}" stroke="black" stroke-width="1.5" opacity="0.9" data-building="{building_id}" data-threshold="{threshold_id}" data-predicted="{pred_class}"/>\n'
            elif building_id == 'b3':  # diamond
                svg_content += f'    <polygon points="{x:.2f},{y-6:.2f} {x+6:.2f},{y:.2f} {x:.2f},{y+6:.2f} {x-6:.2f},{y:.2f}" fill="{color}" stroke="black" stroke-width="1.5" opacity="0.9" data-building="{building_id}" data-threshold="{threshold_id}" data-predicted="{pred_class}"/>\n'
            elif building_id == 'b4':  # triangle down
                svg_content += f'    <polygon points="{x:.2f},{y+6:.2f} {x-6:.2f},{y-5:.2f} {x+6:.2f},{y-5:.2f}" fill="{color}" stroke="black" stroke-width="1.5" opacity="0.9" data-building="{building_id}" data-threshold="{threshold_id}" data-predicted="{pred_class}"/>\n'
    
    svg_content += '  </g>\n'

svg_content += '</svg>'

# Save SVG
svg_path = os.path.join(OUTPUT_DIR, 'tsne_buildings_overlay.svg')
with open(svg_path, 'w') as f:
    f.write(svg_content)

print(f"✓ Saved SVG to: {svg_path}")
print(f"  Total typology points: {len(typology_2d)}")
print(f"  Total building points: {len(building_2d)}")
print()

# Export coordinates as CSV
export_data = []

# Typology data (original positions)
for i in range(len(typology_2d)):
    export_data.append({
        'id': f'typ_{i}',
        'type': 'typology',
        'x': typology_2d[i, 0],
        'y': typology_2d[i, 1],
        'typology': typology_names_list[i],
        'building_id': '',
        'threshold_id': '',
        'predicted_class': typology_names_list[i]
    })

# Building data (interpolated positions)
for i in range(len(building_2d)):
    export_data.append({
        'id': f'bld_{i}',
        'type': 'building',
        'x': building_2d[i, 0],
        'y': building_2d[i, 1],
        'typology': building_predicted_classes[i],
        'building_id': building_ids_list[i],
        'threshold_id': building_threshold_ids[i],
        'predicted_class': building_predicted_classes[i]
    })

export_df = pd.DataFrame(export_data)
csv_path = os.path.join(OUTPUT_DIR, 'tsne_coordinates_with_buildings.csv')
export_df.to_csv(csv_path, index=False)

print(f"✓ Saved coordinates to: {csv_path}")
print()

## Summary Statistics

In [ ]:
print("="*80)
print("SUMMARY")
print("="*80)
print()

print("Original t-SNE (from train_mae_full.ipynb):")
print(f"  Total points: {len(typology_2d)}")
for typ in TYPOLOGY_NAMES:
    count = sum(1 for t in typology_names_list if t == typ)
    print(f"    {typ}: {count} points")
print()

print("Building Classification Summary:")
for building_id in BUILDING_IDS:
    building_preds = [pred for bid, pred in zip(building_ids_list, building_predicted_classes) if bid == building_id]
    if building_preds:
        pred_counts = pd.Series(building_preds).value_counts()
        top_pred = pred_counts.index[0]
        top_count = pred_counts.iloc[0]
        pct = top_count / len(building_preds) * 100
        print(f"  {building_id.upper()}: {len(building_preds)} thresholds")
        print(f"    Top: {top_pred} ({top_count}/{len(building_preds)} = {pct:.1f}%)")
        print(f"    Distribution: {dict(pred_counts.head(3))}")
print()

print("Output Files:")
print(f"  PNG: {os.path.join(OUTPUT_DIR, 'tsne_buildings_overlay.png')}")
print(f"  SVG: {os.path.join(OUTPUT_DIR, 'tsne_buildings_overlay.svg')}")
print(f"  CSV: {os.path.join(OUTPUT_DIR, 'tsne_coordinates_with_buildings.csv')}")
print()

print("SVG Structure:")
print("  - Original typologies: circles with data-typology attribute")
print("  - Buildings: shapes (square/triangle/diamond) with:")
print("      data-building: building ID (b1/b2/b3/b4)")
print("      data-threshold: threshold ID")
print("      data-predicted: predicted typology class")
print("  - Colors match predicted typology")
print()
print("="*80)
print("\n✓ Visualization complete!")